# Self-Healing Digital Twins: Repair-for-the-Twin Evaluation with Uncertainty-Aware Imputation


## Hypotheses
- **H1 (decision-layer divergence).** Imputation-RMSE rankings track twin *accuracy* rankings reasonably
  well, but **diverge at the decision layer**: τ(RMSE, decision regret) is low/non-significant while
  τ(RMSE, PHM08) stays high. The failure of reconstruction metrics is *invisible* until a cost model
  is attached.
- **H2 (uncertainty-aware repair).** Propagating imputation uncertainty (multiple imputation × ensemble,
  Rubin's rules) materially raises interval coverage and — via a risk-aware alarm policy — cuts decision
  cost several-fold vs point-imputation pipelines. One-sided paired Wilcoxon across 10 seeds.
- **H3 (repair-for-the-twin criterion).** Selecting the imputer by downstream risk-aware decision cost
  beats selecting by imputation RMSE in out-of-sample maintenance cost.

## Variance decomposition (methodological core)
$$\operatorname{Var}(\mathrm{RUL}_t) = \underbrace{\mathbb{E}[\operatorname{Var}(\mathrm{RUL}_t \mid Z_t)]}_{\bar W\text{: aleatoric + epistemic (within)}} + \underbrace{\operatorname{Var}(\mathbb{E}[\mathrm{RUL}_t \mid Z_t])}_{B\text{: imputation (between)}}, \qquad T = \bar W + (1 + 1/m)B.$$

**v2 upgrades over v1:** (i) *heteroscedastic* twin head (μ, log σ²; Gaussian NLL) so $\bar W$ contains
aleatoric variance — v1's ensemble-only $\bar W$ under-covered everywhere; (ii) split-**conformal
multiplier** calibrated on a clean validation split (connects directly to DQ-Twin-Bench: clean-calibrated
conformal degrades under corruption; the $B$ term is the partial remedy); (iii) 10 seeds + pre-registered
**one-sided** Wilcoxon (H2 is directional); (iv) US-GAN trained to 150 epochs; (v) **true B-term ablation**
computed inside the harness; (vi) saved per-window predictions enabling post-hoc **cost-ratio sensitivity**
without re-running; (vii) **Google Drive checkpoint/resume** for every heavy component.

> **Resume behaviour:** if the Colab runtime disconnects, just *Run all* again — trained models reload
> from Drive, completed (dataset × mechanism × rate × seed) result blocks are skipped, and the harness
> continues from the first missing block.

In [ ]:
# ============================================================
# Section 1: Configuration & deterministic seeding
# ============================================================
!pip -q install "pypots==1.5" "torch>=2.3" "scikit-learn>=1.5" \
    pandas numpy matplotlib seaborn scipy tqdm h5py

import json
import os
import random

import numpy as np
import torch

GLOBAL_SEED = 2026

# --------------- experiment scale ---------------
QUICK_RUN = False                       # True => smoke test (~25 min, FD001)
SUBSETS_TO_RUN = (["FD001"] if QUICK_RUN
                  else ["FD001", "FD002", "FD003", "FD004"])
ANCHOR = "FD001"                        # subset used for headline figures

ENSEMBLE_K = 3 if QUICK_RUN else 5
TWIN_EPOCHS = 15 if QUICK_RUN else 60
IMPUTER_EPOCHS = 8 if QUICK_RUN else 40      # SAITS
USGAN_EPOCHS = 10 if QUICK_RUN else 150      # v2 fix: GAN needs longer
CSDI_EPOCHS = 10 if QUICK_RUN else 60
MI_DRAWS_CSDI = 5 if QUICK_RUN else 20
MI_DRAWS_MICE = 3 if QUICK_RUN else 5
SEEDS = [0, 1] if QUICK_RUN else list(range(10))   # v2 fix: 10 seeds
RATES = [0.3] if QUICK_RUN else [0.1, 0.3, 0.5]
MECHANISMS = ["MCAR", "MNAR"] if QUICK_RUN else ["MCAR", "MAR", "MNAR"]
GAP_LEN = 3
NOMINAL = 0.95                          # target interval coverage

# v2 fix: session-stable mechanism ids (never use hash() -- it is salted
# per interpreter session and would break resume determinism).
MECH_ID = {"MCAR": 1, "MAR": 2, "MNAR": 3}


def set_global_determinism(seed: int = GLOBAL_SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        print(f"[warn] deterministic algorithms partial: {exc}")


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_global_determinism()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
GENERATOR = torch.Generator().manual_seed(GLOBAL_SEED)

if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} "
          f"GB | BF16: {torch.cuda.is_bf16_supported()}")
else:
    print("[warn] No GPU -- switch the Colab runtime to L4.")

---
## Section 2 — Google Drive Persistence Layer (checkpoint + resume)

Everything heavy is cached under `MyDrive/self_healing_twins/`:

```
self_healing_twins/
├── models/{FD00x}/            twin_0..K-1.pt, histories.json, conformal.json,
│                              saits.pypots, usgan.pypots, csdi.pypots
├── results/all_results.csv    incrementally appended, de-duplicated on key
├── preds/{FD00x}_{mech}_{rate}_{seed}.npz   per-window μ, σ² per imputer
└── figures/                   all publication PDFs
```

If Drive can't be mounted (local testing), everything falls back to `/content/ckpt_local` — the notebook
still runs, it just won't survive a disconnect.

In [ ]:
# ============================================================
# Section 2: Drive mount + persistence helpers
# ============================================================
from pathlib import Path

try:
    from google.colab import drive as _gdrive
    _gdrive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/self_healing_twins")
    print("[drive] mounted -- checkpoints will survive disconnects.")
except Exception as exc:
    BASE = Path("/content/ckpt_local")
    print(f"[warn] Drive unavailable ({exc}); using local fallback {BASE}")

MODELS_DIR = BASE / "models"
RESULTS_CSV = BASE / "results" / "all_results.csv"
PREDS_DIR = BASE / "preds"
FIGS_DIR = BASE / "figures"
for _p in (MODELS_DIR, RESULTS_CSV.parent, PREDS_DIR, FIGS_DIR):
    _p.mkdir(parents=True, exist_ok=True)

import pandas as pd

KEY_COLS = ["dataset", "mech", "rate", "seed", "imputer"]


def load_results() -> pd.DataFrame:
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def append_results(rows: list) -> pd.DataFrame:
    """Append rows, de-duplicate on the experiment key, atomic write."""
    df = pd.concat([load_results(), pd.DataFrame(rows)], ignore_index=True)
    df = df.drop_duplicates(subset=KEY_COLS, keep="last")
    tmp = RESULTS_CSV.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    tmp.replace(RESULTS_CSV)
    return df


def block_done(dataset, mech, rate, seed, imputers) -> bool:
    """A (dataset, mech, rate, seed) block is complete iff every imputer row
    exists AND the prediction archive is on disk."""
    df = load_results()
    if df.empty:
        return False
    sub = df[(df.dataset == dataset) & (df.mech == mech)
             & (df.rate == rate) & (df.seed == seed)]
    return (set(imputers) <= set(sub.imputer)
            and preds_path(dataset, mech, rate, seed).exists())


def preds_path(dataset, mech, rate, seed) -> Path:
    return PREDS_DIR / f"{dataset}_{mech}_{int(rate*100)}_{seed}.npz"


def mpath(dataset, name) -> Path:
    d = MODELS_DIR / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d / name

---
## Section 3 — Data Ingestion & Preprocessing (DQ-Twin-Bench conventions)

SHA-256-verified download (the zip itself is also cached on Drive so reconnects skip the network);
14 informative sensors; window 30; piecewise RUL cap 125; min-max (FD001/FD003) or **k-means k=6
condition-based** min-max (FD002/FD004). Missingness (Rubin 1976: MCAR/MAR/MNAR, burst gaps of
`GAP_LEN=3`) is injected **on the test/inference stream only** with a **session-stable** RNG so resumed
runs reproduce identical masks.

In [ ]:
# ============================================================
# Section 3a: Robust ingestion (Drive-cached zip, SHA256, fallbacks)
# ============================================================
import hashlib
import io
import urllib.request
import zipfile

DATA_DIR = Path("/content/cmapss")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_CACHE = BASE / "cmapss.zip"

MIRRORS = [
    "https://phm-datasets.s3.amazonaws.com/NASA/"
    "6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]
EXPECTED_FILES = [f"{s}_FD00{i}.txt" for i in range(1, 5)
                  for s in ("train", "test")]
EXPECTED_FILES += [f"RUL_FD00{i}.txt" for i in range(1, 5)]


def _extract(blob: bytes) -> None:
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        zf.extractall(DATA_DIR)
    for inner in list(DATA_DIR.rglob("*.zip")):
        with zipfile.ZipFile(inner) as zf:
            zf.extractall(DATA_DIR)
        inner.unlink()
    for txt in DATA_DIR.rglob("*FD00*.txt"):
        if txt.parent != DATA_DIR:
            txt.replace(DATA_DIR / txt.name)


def fetch_cmapss() -> None:
    if all((DATA_DIR / f).exists() for f in EXPECTED_FILES):
        print("[cache] C-MAPSS present in runtime.")
        return
    if ZIP_CACHE.exists():
        print("[drive] restoring zip from Drive cache.")
        _extract(ZIP_CACHE.read_bytes())
        if all((DATA_DIR / f).exists() for f in EXPECTED_FILES):
            return
    last = None
    for url in MIRRORS:
        try:
            print(f"[fetch] {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab"})
            blob = urllib.request.urlopen(req, timeout=180).read()
            print(f"[sha256] {hashlib.sha256(blob).hexdigest()} "
                  f"({len(blob)/1e6:.1f} MB)")
            ZIP_CACHE.write_bytes(blob)          # cache for future sessions
            _extract(blob)
            if all((DATA_DIR / f).exists() for f in EXPECTED_FILES):
                print("[ok] all FD001-FD004 files verified.")
                return
        except Exception as exc:
            last = exc
            print(f"[warn] mirror failed: {exc}")
    raise RuntimeError(f"All mirrors failed: {last}")


fetch_cmapss()

In [ ]:
# ============================================================
# Section 3b: Preprocessing + corruption injection
# ============================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

COL_NAMES = (["unit", "cycle"] + [f"op{i}" for i in range(1, 4)]
             + [f"s{i}" for i in range(1, 22)])
KEEP_SENSORS = [2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21]
FEATURE_COLS = [f"s{i}" for i in KEEP_SENSORS]
N_FEAT = len(FEATURE_COLS)
WINDOW = 30
RUL_CAP = 125
MULTI_REGIME = {"FD002", "FD004"}


def load_subset(name):
    train = pd.read_csv(DATA_DIR / f"train_{name}.txt", sep=r"\s+",
                        header=None, names=COL_NAMES)
    test = pd.read_csv(DATA_DIR / f"test_{name}.txt", sep=r"\s+",
                       header=None, names=COL_NAMES)
    rul = pd.read_csv(DATA_DIR / f"RUL_{name}.txt", sep=r"\s+",
                      header=None).values.ravel()
    return train, test, rul


def add_piecewise_rul(df):
    max_cycle = df.groupby("unit")["cycle"].transform("max")
    df = df.copy()
    df["RUL"] = (max_cycle - df["cycle"]).clip(upper=RUL_CAP)
    return df


def condition_normalize(train, test, name):
    train, test = train.copy(), test.copy()
    if name in MULTI_REGIME:
        km = KMeans(n_clusters=6, n_init=10, random_state=GLOBAL_SEED)
        train["regime"] = km.fit_predict(train[["op1", "op2", "op3"]])
        test["regime"] = km.predict(test[["op1", "op2", "op3"]])
        for reg in range(6):
            sc = MinMaxScaler()
            m_tr, m_te = train["regime"] == reg, test["regime"] == reg
            sc.fit(train.loc[m_tr, FEATURE_COLS])
            train.loc[m_tr, FEATURE_COLS] = sc.transform(
                train.loc[m_tr, FEATURE_COLS])
            if m_te.any():
                test.loc[m_te, FEATURE_COLS] = sc.transform(
                    test.loc[m_te, FEATURE_COLS])
    else:
        sc = MinMaxScaler().fit(train[FEATURE_COLS])
        train[FEATURE_COLS] = sc.transform(train[FEATURE_COLS])
        test[FEATURE_COLS] = sc.transform(test[FEATURE_COLS])
    return train, test


def make_windows(df, is_test, rul_vec=None):
    xs, ys = [], []
    for uid, grp in df.groupby("unit"):
        arr = grp[FEATURE_COLS].values
        if is_test:
            if len(arr) < WINDOW:
                arr = np.pad(arr, ((WINDOW - len(arr), 0), (0, 0)), "edge")
            xs.append(arr[-WINDOW:])
            ys.append(min(rul_vec[int(uid) - 1], RUL_CAP))
        else:
            rul = grp["RUL"].values
            for i in range(len(arr) - WINDOW + 1):
                xs.append(arr[i:i + WINDOW])
                ys.append(rul[i + WINDOW - 1])
    return np.asarray(xs, np.float32), np.asarray(ys, np.float32)


def inject_missing(x, mechanism="MCAR", rate=0.2, gap_len=1, rng=None):
    """Rubin (1976) mechanisms; returns (x with NaNs, injected-position mask).
    Callers MUST seed rng via mech/rate/seed integers only (resume-stable)."""
    rng = rng or np.random.default_rng(GLOBAL_SEED)
    xc = x.copy()
    n, t, f = xc.shape
    mask = np.zeros_like(xc, dtype=bool)
    for i in range(n):
        for j in range(f):
            if mechanism == "MCAR":
                p = np.full(t, rate)
            elif mechanism == "MAR":
                driver = xc[i, :, (j + 1) % f]
                p = rate * 2.0 * (driver > np.median(driver))
            elif mechanism == "MNAR":
                p = rate * 2.0 * (xc[i, :, j] > np.median(xc[i, :, j]))
            else:
                raise ValueError(mechanism)
            starts = np.where(rng.random(t) < p / max(gap_len, 1))[0]
            for s in starts:
                mask[i, s:min(s + gap_len, t), j] = True
    xc[mask] = np.nan
    return xc, mask


def corruption_rng(mech, rate, seed):
    """Session-stable RNG (v2 fix: no hash())."""
    return np.random.default_rng(
        100000 * MECH_ID[mech] + int(rate * 1000) * 100 + seed)


DATASETS = {}
for _name in SUBSETS_TO_RUN:
    _tr, _te, _rul = load_subset(_name)
    _tr = add_piecewise_rul(_tr)
    _tr, _te = condition_normalize(_tr, _te, _name)
    _xtr, _ytr = make_windows(_tr, is_test=False)
    _xte, _yte = make_windows(_te, is_test=True, rul_vec=_rul)
    DATASETS[_name] = dict(x_tr=_xtr, y_tr=_ytr, x_te=_xte, y_te=_yte)
    print(f"{_name}: train {_xtr.shape}  test {_xte.shape}")

---
## Section 4 — Methodology: Uncertainty-Aware Repair → Heteroscedastic Twin → Rubin → Conformal

**Pipeline per test stream:** corrupt → repair (9-imputer spectrum) → heteroscedastic deep-ensemble twin →
(for generative imputers) Rubin pooling over $m$ posterior draws → conformal-scaled interval →
risk-aware maintenance policy.

**Why the v2 twin is heteroscedastic.** v1's within-term $\bar W$ used ensemble disagreement only —
epistemic variance with zero aleatoric content — and *every* pipeline under-covered badly (oracle
coverage 0.21). v2's twin predicts $(\mu, \log\sigma^2)$ trained with Gaussian NLL, so each member
carries its own aleatoric estimate. The ensemble is treated as a uniform mixture:
$\mu_\* = \tfrac1K\sum_k \mu_k$, $\;\sigma_\*^2 = \tfrac1K\sum_k \sigma_k^2 + \operatorname{Var}_k(\mu_k)$
(aleatoric + epistemic). Rubin's rules then add the imputation term across draws:
$T = \bar W + (1+1/m)B$.

**Why a conformal multiplier.** Even a well-specified NLL model miscalibrates in finite samples. We apply
**split-conformal scaling**: on a *clean* validation split, compute normalized scores
$s_i = |y_i - \mu_i| / \sigma_i$ and take $\hat q$ = the $\lceil (n+1)(1-\alpha) \rceil / n$ empirical
quantile; intervals are $\mu \pm \hat q\,\sigma$. On clean data this guarantees ≈95% coverage. Under
corruption, exchangeability with the calibration set breaks — **exactly the DQ-Twin-Bench failure mode** —
and the B-term is the measured partial remedy. The manuscript claim is therefore *"the B term recovers a
large fraction of the coverage lost to corruption and converts it into multi-fold decision-cost
reduction"*, not "restores nominal coverage".

**Imputer spectrum:** `no_repair` · `mean` · `ffill` · `KNN` · `MICE` (MI, $m{=}5$) · `US-GAN` (150 ep) ·
`SAITS` · `CSDI` (MI, $m{=}20$ diffusion posterior draws) · `oracle_clean`.

In [ ]:
# ============================================================
# Section 4a: Imputer spectrum with Drive-cached deep models
# ============================================================
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer

from pypots.imputation import CSDI, SAITS, USGAN


def _flatten(x):
    n, t, f = x.shape
    return x.reshape(n * t, f).copy(), (n, t, f)


def _unflatten(x2d, shape):
    return x2d.reshape(*shape).astype(np.float32)


def impute_mean(xc):
    flat, sh = _flatten(xc)
    col_mean = np.nanmean(flat, axis=0)
    idx = np.where(np.isnan(flat))
    flat[idx] = np.take(col_mean, idx[1])
    return _unflatten(flat, sh)


def impute_ffill(xc):
    out = xc.copy()
    for i in range(out.shape[0]):
        out[i] = pd.DataFrame(out[i]).ffill().bfill().values
    return np.nan_to_num(out, nan=0.0).astype(np.float32)


def impute_knn(xc, k=5):
    flat, sh = _flatten(xc)
    return _unflatten(KNNImputer(n_neighbors=k).fit_transform(flat), sh)


def impute_mice_multi(xc, m=5):
    """m posterior draws via IterativeImputer(sample_posterior=True) under
    m different seeds (the scikit-learn-endorsed MI recipe). [n, m, t, f]."""
    flat, sh = _flatten(xc)
    draws = []
    for d in range(m):
        imp = IterativeImputer(sample_posterior=True, max_iter=10,
                               random_state=GLOBAL_SEED + d)
        draws.append(_unflatten(imp.fit_transform(flat), sh))
    return np.stack(draws, axis=1)


def _fit_or_load(model, ckpt: Path, x_tr, label: str):
    """PyPOTS models: reload from Drive if a checkpoint exists."""
    if ckpt.exists():
        try:
            model.load(str(ckpt))
            print(f"[resume] {label} loaded from {ckpt.name}")
            return model
        except Exception as exc:
            print(f"[warn] {label} load failed ({exc}); retraining.")
    model.fit({"X": x_tr})
    try:
        model.save(str(ckpt), overwrite=True)
        print(f"[ckpt] {label} saved -> {ckpt.name}")
    except TypeError:                      # older signature without overwrite
        model.save(str(ckpt))
    return model


def get_saits(dataset, x_tr):
    m = SAITS(n_steps=WINDOW, n_features=N_FEAT, n_layers=2, d_model=256,
              n_heads=4, d_k=64, d_v=64, d_ffn=128, dropout=0.1,
              epochs=IMPUTER_EPOCHS, patience=5, batch_size=256,
              num_workers=2, device=DEVICE)
    return _fit_or_load(m, mpath(dataset, "saits.pypots"), x_tr, "SAITS")


def get_usgan(dataset, x_tr):
    """US-GAN (Miao et al., AAAI 2021): GAIN-style hint-rate adversarial
    imputer with a BRITS generator. v2 fix: 150 epochs (GAN convergence)."""
    m = USGAN(n_steps=WINDOW, n_features=N_FEAT, rnn_hidden_size=128,
              hint_rate=0.7, lambda_mse=1.0, epochs=USGAN_EPOCHS,
              patience=15, batch_size=256, num_workers=2, device=DEVICE)
    return _fit_or_load(m, mpath(dataset, "usgan.pypots"), x_tr, "US-GAN")


def get_csdi(dataset, x_tr):
    m = CSDI(n_steps=WINDOW, n_features=N_FEAT, n_layers=4, n_heads=8,
             n_channels=64, d_time_embedding=64, d_feature_embedding=32,
             d_diffusion_embedding=128, n_diffusion_steps=50,
             target_strategy="random", schedule="quad", epochs=CSDI_EPOCHS,
             patience=8, batch_size=64, num_workers=2, device=DEVICE)
    return _fit_or_load(m, mpath(dataset, "csdi.pypots"), x_tr, "CSDI")


def impute_point(model, xc):
    return model.impute({"X": xc}).astype(np.float32)


def impute_csdi_multi(model, xc, m=20):
    out = model.predict({"X": xc}, n_sampling_times=m)["imputation"]
    out = np.asarray(out, dtype=np.float32)
    assert out.ndim == 4 and out.shape[1] == m, f"unexpected {out.shape}"
    return out                                  # [n, m, t, f]

In [ ]:
# ============================================================
# Section 4b: Heteroscedastic twin + mixture ensemble + Rubin + conformal
# ============================================================
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


class RULTwin(nn.Module):
    """1D-CNN + GRU twin with a heteroscedastic head: outputs (mu, log_var).
    v2 fix: aleatoric variance enters the within-term W of Rubin's rules."""

    def __init__(self, n_feat: int = 14, hid: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_feat, hid, 5, padding=2), nn.GELU(),
            nn.Conv1d(hid, hid, 5, padding=2), nn.GELU())
        self.gru = nn.GRU(hid, hid, batch_first=True, num_layers=2,
                          dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(hid, 2))

    def forward(self, x):
        h = self.conv(x.transpose(1, 2)).transpose(1, 2)
        _, hn = self.gru(h)
        out = self.head(hn[-1])
        mu, log_var = out[:, 0], out[:, 1].clamp(-6.0, 8.0)
        return mu, log_var


def gaussian_nll(mu, log_var, y):
    return 0.5 * (log_var + (y - mu) ** 2 / log_var.exp()).mean()


@torch.no_grad()
def ensemble_predict(members, x, batch=4096):
    """Uniform-mixture ensemble: mean, total within-variance
    (aleatoric mean-of-sigma^2 + epistemic var-of-mu)."""
    mus, sig2s = [], []
    for m in members:
        m.eval()
        mu_o, s2_o = [], []
        for i in range(0, len(x), batch):
            xt = torch.tensor(x[i:i + batch]).to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=DTYPE,
                                enabled=DEVICE == "cuda"):
                mu, lv = m(xt)
            mu_o.append(mu.float().cpu().numpy())
            s2_o.append(lv.float().exp().cpu().numpy())
        mus.append(np.concatenate(mu_o))
        sig2s.append(np.concatenate(s2_o))
    mus, sig2s = np.stack(mus), np.stack(sig2s)          # [K, n]
    mean = mus.mean(0)
    var = sig2s.mean(0) + mus.var(0, ddof=1)
    return mean, var


def rubin_pool(draw_means, draw_vars):
    """draw_means/draw_vars: [m, n] per-imputation ensemble outputs.
    T = W_bar + (1 + 1/m) * B. Also returns W_bar alone for the B-ablation."""
    m = draw_means.shape[0]
    pooled = draw_means.mean(0)
    w_bar = draw_vars.mean(0)
    b = draw_means.var(0, ddof=1) if m > 1 else np.zeros_like(pooled)
    total = w_bar + (1.0 + 1.0 / m) * b
    return pooled, total, w_bar


def conformal_multiplier(members, x_val, y_val, alpha=1 - NOMINAL):
    """Split-conformal q on CLEAN validation: s = |y - mu| / sigma;
    q = ceil((n+1)(1-alpha))/n empirical quantile of s."""
    mu, var = ensemble_predict(members, x_val)
    s = np.abs(y_val - mu) / np.sqrt(np.maximum(var, 1e-9))
    n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    q = float(np.sort(s)[min(k, n) - 1])
    return q

---
## Section 5 — Production Training Loop (BF16, cosine + warmup, MSE→NLL curriculum, checkpointed)

Each ensemble member trains with **MSE for the first 20% of epochs** (stabilises μ before the variance
head engages), then **Gaussian NLL**; cosine annealing with 5% linear warmup; gradient clipping 1.0;
early stopping (patience 8) on validation RMSE; **per-member checkpoints on Drive** — reconnects reload
in seconds. The conformal multiplier $\hat q$ is computed once per dataset on the clean validation split
and cached in `conformal.json`.

In [ ]:
# ============================================================
# Section 5: Training with Drive checkpoint/resume
# ============================================================
import copy
import math


def make_loader(x, y, bs=512, shuffle=True):
    ds = TensorDataset(torch.tensor(x), torch.tensor(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=4,
                      pin_memory=True, persistent_workers=True,
                      prefetch_factor=4, worker_init_fn=seed_worker,
                      generator=GENERATOR, drop_last=shuffle)


def cosine_warmup(step, total, warmup, base_lr):
    if step < warmup:
        return base_lr * step / max(1, warmup)
    prog = (step - warmup) / max(1, total - warmup)
    return 0.5 * base_lr * (1 + math.cos(math.pi * prog))


def train_member(x_tr, y_tr, x_val, y_val, seed, ckpt: Path,
                 epochs=None, base_lr=1e-3):
    epochs = epochs or TWIN_EPOCHS
    torch.manual_seed(GLOBAL_SEED + seed)
    model = RULTwin(n_feat=x_tr.shape[-1]).to(DEVICE)
    if ckpt.exists():
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model"])
        print(f"[resume] twin member {seed} "
              f"(val RMSE {state['best_val']:.3f}) from Drive")
        return model, state["best_val"], state["history"]

    opt = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=1e-4)
    loader = make_loader(x_tr, y_tr)
    total_steps = epochs * len(loader)
    warmup = int(0.05 * total_steps)
    mse_epochs = max(1, int(0.2 * epochs))       # curriculum: MSE -> NLL
    mse = nn.MSELoss()

    best_val, best_state, bad, step, history = float("inf"), None, 0, 0, []
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            for g in opt.param_groups:
                g["lr"] = cosine_warmup(step, total_steps, warmup, base_lr)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=DTYPE,
                                enabled=DEVICE == "cuda"):
                mu, lv = model(xb)
                loss = (mse(mu, yb) if ep < mse_epochs
                        else gaussian_nll(mu, lv, yb))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step += 1
        mu_v, _ = ensemble_predict([model], x_val)
        vrmse = float(np.sqrt(np.mean((mu_v - y_val) ** 2)))
        history.append(vrmse)
        if vrmse < best_val:
            best_val = vrmse
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
            if bad >= 8 and ep >= mse_epochs + 2:
                break
    model.load_state_dict(best_state)
    torch.save({"model": best_state, "best_val": best_val,
                "history": history}, ckpt)
    print(f"[ckpt] twin member {seed} saved (val RMSE {best_val:.3f})")
    return model, best_val, history


def get_twin(dataset, x_tr, y_tr, k=None, val_frac=0.1):
    """Train-or-load the K-member ensemble + conformal multiplier."""
    k = k or ENSEMBLE_K
    idx = np.random.default_rng(GLOBAL_SEED).permutation(len(x_tr))
    cut = int(len(x_tr) * (1 - val_frac))
    tr_i, val_i = idx[:cut], idx[cut:]
    members, histories = [], []
    for s in range(k):
        m, _, h = train_member(x_tr[tr_i], y_tr[tr_i],
                               x_tr[val_i], y_tr[val_i],
                               seed=s, ckpt=mpath(dataset, f"twin_{s}.pt"))
        members.append(m)
        histories.append(h)
    json.dump(histories, open(mpath(dataset, "histories.json"), "w"))

    cpath = mpath(dataset, "conformal.json")
    if cpath.exists():
        q = json.load(open(cpath))["q"]
        print(f"[resume] conformal q = {q:.3f}")
    else:
        q = conformal_multiplier(members, x_tr[val_i], y_tr[val_i])
        json.dump({"q": q}, open(cpath, "w"))
        print(f"[conformal] q = {q:.3f} "
              f"(clean-val calibrated; 1.96 would be the Gaussian value)")
    return members, histories, q

---
## Section 6 — Rigorous Evaluation: Metrics, Resumable Harness, Statistics

Per (dataset × mechanism × rate × seed × imputer) we log: imputation RMSE/MAE (mask-held-out),
RUL RMSE, PHM08, conformal-interval coverage & width, threshold-policy **decision regret**,
**risk-aware regret** (alarm iff lower bound ≤ τ), and — for the MI imputers — the same interval and
risk metrics with the **B term removed** ($T = \bar W$ only): the *true in-harness ablation*.

Raw per-window predictions (μ, σ²_total, σ²_W-only) are archived to Drive per block, enabling all
post-hoc analyses (cost-ratio sweeps, regret decomposition, calibration curves) **without re-running**.

**Statistics (pre-registered):** H1 via Kendall's τ between imputation-RMSE and regret rankings with
**seed-bootstrap 95% CIs**; H2 via **one-sided** paired Wilcoxon (CSDI-Rubin < SAITS-point on risk-aware
regret; directional hypothesis, n = 10 seeds); H3 via the selection-criterion cost comparison.

In [ ]:
# ============================================================
# Section 6a: Metric definitions
# ============================================================
from scipy.stats import kendalltau, spearmanr, wilcoxon

COST = dict(C_pm=1.0, C_cm=10.0, C_fa=2.0)
TAU = 20.0
IMPUTERS = ["no_repair", "mean", "ffill", "knn", "mice", "usgan",
            "saits", "csdi", "oracle_clean"]
REAL_IMPUTERS = [i for i in IMPUTERS
                 if i not in ("no_repair", "oracle_clean")]


def imputation_rmse(x_true, x_imp, mask):
    return float(np.sqrt(np.mean((x_true[mask] - x_imp[mask]) ** 2)))


def imputation_mae(x_true, x_imp, mask):
    return float(np.mean(np.abs(x_true[mask] - x_imp[mask])))


def rul_rmse(pred, true):
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def phm08_score(pred, true):
    d = pred - true
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0,
                                 np.exp(d / 10.0) - 1.0)))


def interval_metrics(mean, var, true, q):
    half = q * np.sqrt(np.maximum(var, 1e-9))
    lo, hi = mean - half, mean + half
    return (float(np.mean((true >= lo) & (true <= hi))),
            float(np.mean(hi - lo)))


def cost_from_alarm(alarm, true, tau=TAU, cost=COST):
    crit = true <= tau
    return float(cost["C_pm"] * (alarm & crit).sum()
                 + cost["C_cm"] * ((~alarm) & crit).sum()
                 + cost["C_fa"] * (alarm & ~crit).sum())


def decision_regret(pred, true, tau=TAU, cost=COST):
    return (cost_from_alarm(pred <= tau, true, tau, cost)
            - cost_from_alarm(true <= tau, true, tau, cost))


def risk_aware_regret(mean, var, true, q, tau=TAU, cost=COST):
    alarm = (mean - q * np.sqrt(np.maximum(var, 1e-9))) <= tau
    return (cost_from_alarm(alarm, true, tau, cost)
            - cost_from_alarm(true <= tau, true, tau, cost))

In [ ]:
# ============================================================
# Section 6b: Resumable multi-dataset harness (+ true B-ablation,
#             + prediction archiving for post-hoc analyses)
# ============================================================
def repair(imp, xc, models):
    """Return (x_repaired, draws-or-None)."""
    if imp == "no_repair":
        return np.nan_to_num(xc, nan=0.0).astype(np.float32), None
    if imp == "mean":
        return impute_mean(xc), None
    if imp == "ffill":
        return impute_ffill(xc), None
    if imp == "knn":
        return impute_knn(xc), None
    if imp == "usgan":
        return impute_point(models["usgan"], xc), None
    if imp == "saits":
        return impute_point(models["saits"], xc), None
    if imp == "mice":
        draws = impute_mice_multi(xc, m=MI_DRAWS_MICE)
        return draws.mean(axis=1), draws
    if imp == "csdi":
        draws = impute_csdi_multi(models["csdi"], xc, m=MI_DRAWS_CSDI)
        return np.median(draws, axis=1), draws
    raise ValueError(imp)


def evaluate_block(dataset, mech, rate, seed, x_te, y_te, twin, models, q):
    """One (mech, rate, seed) block: all imputers -> rows + preds archive."""
    rng = corruption_rng(mech, rate, seed)
    xc, mask = inject_missing(x_te, mech, rate, gap_len=GAP_LEN, rng=rng)
    rows, arch = [], {"y": y_te, "mask_rate": np.float32(mask.mean())}
    for imp in IMPUTERS:
        if imp == "oracle_clean":
            xr, draws = x_te, None
        else:
            xr, draws = repair(imp, xc, models)

        if imp in ("no_repair", "oracle_clean"):
            i_rmse = i_mae = np.nan
        else:
            i_rmse = imputation_rmse(x_te, xr, mask)
            i_mae = imputation_mae(x_te, xr, mask)

        if draws is not None:                       # Rubin path (MI)
            d_means, d_vars = [], []
            for k in range(draws.shape[1]):
                mu, v = ensemble_predict(twin, draws[:, k])
                d_means.append(mu)
                d_vars.append(v)
            mean, var, w_only = rubin_pool(np.stack(d_means),
                                           np.stack(d_vars))
        else:
            mean, var = ensemble_predict(twin, xr)
            w_only = None

        cov, wid = interval_metrics(mean, var, y_te, q)
        row = dict(dataset=dataset, mech=mech, rate=rate, seed=seed,
                   imputer=imp, imp_rmse=i_rmse, imp_mae=i_mae,
                   rul_rmse=rul_rmse(mean, y_te),
                   phm08=phm08_score(mean, y_te),
                   coverage=cov, width=wid,
                   regret=decision_regret(mean, y_te),
                   regret_risk=risk_aware_regret(mean, var, y_te, q),
                   coverage_noB=np.nan, width_noB=np.nan,
                   regret_risk_noB=np.nan)
        arch[f"{imp}_mean"] = mean.astype(np.float32)
        arch[f"{imp}_var"] = var.astype(np.float32)
        if w_only is not None:                      # true B-term ablation
            cov0, wid0 = interval_metrics(mean, w_only, y_te, q)
            row.update(coverage_noB=cov0, width_noB=wid0,
                       regret_risk_noB=risk_aware_regret(
                           mean, w_only, y_te, q))
            arch[f"{imp}_var_noB"] = w_only.astype(np.float32)
        rows.append(row)
    np.savez_compressed(preds_path(dataset, mech, rate, seed), **arch)
    return rows


def run_dataset(name):
    d = DATASETS[name]
    x_tr, y_tr, x_te, y_te = d["x_tr"], d["y_tr"], d["x_te"], d["y_te"]
    print(f"\n===== {name}: models (train-or-load from Drive) =====")
    twin, hist, q = get_twin(name, x_tr, y_tr)
    models = dict(saits=get_saits(name, x_tr),
                  usgan=get_usgan(name, x_tr),
                  csdi=get_csdi(name, x_tr))
    total = len(MECHANISMS) * len(RATES) * len(SEEDS)
    done = 0
    for mech in MECHANISMS:
        for rate in RATES:
            for seed in SEEDS:
                done += 1
                if block_done(name, mech, rate, seed, IMPUTERS):
                    print(f"[skip {done}/{total}] {name} {mech} "
                          f"r={rate} s={seed} (already on Drive)")
                    continue
                rows = evaluate_block(name, mech, rate, seed,
                                      x_te, y_te, twin, models, q)
                append_results(rows)               # flush after every block
                print(f"[done {done}/{total}] {name} {mech} "
                      f"r={rate} s={seed}")
    return hist, q


TWIN_HISTORIES, CONFORMAL_Q = {}, {}
for _name in SUBSETS_TO_RUN:
    TWIN_HISTORIES[_name], CONFORMAL_Q[_name] = run_dataset(_name)

results = load_results()
results = results[results.dataset.isin(SUBSETS_TO_RUN)]
print(f"\nTotal result rows: {len(results)}")
results.groupby(["dataset", "imputer"])[
    ["imp_rmse", "rul_rmse", "phm08", "coverage", "regret",
     "regret_risk"]].mean().round(3)

In [ ]:
# ============================================================
# Section 6c: Statistics -- divergence + bootstrap CIs, one-sided
#             Wilcoxon (n=10), seed CIs, H3 criterion
# ============================================================
def _tau_from_agg(agg):
    t, p = kendalltau(agg.imp_rmse.rank(), agg.regret.rank())
    return t, p


def ranking_divergence(df, dataset=None, n_boot=2000, rng_seed=0):
    """Kendall tau (imp-RMSE vs regret) per (mech, rate) with
    seed-bootstrap 95% CIs; plus tau vs PHM08 for the two-layer story."""
    if dataset:
        df = df[df.dataset == dataset]
    real = df[df.imputer.isin(REAL_IMPUTERS)]
    rng = np.random.default_rng(rng_seed)
    out = []
    for (mech, rate), g in real.groupby(["mech", "rate"]):
        agg = g.groupby("imputer")[["imp_rmse", "regret", "phm08"]].mean()
        tau, p = _tau_from_agg(agg)
        tau_phm, _ = kendalltau(agg.imp_rmse.rank(), agg.phm08.rank())
        rho, _ = spearmanr(agg.imp_rmse, agg.regret)
        boots = []
        seeds = g.seed.unique()
        for _ in range(n_boot):
            take = rng.choice(seeds, size=len(seeds), replace=True)
            gb = pd.concat([g[g.seed == s] for s in take])
            ab = gb.groupby("imputer")[["imp_rmse", "regret"]].mean()
            boots.append(kendalltau(ab.imp_rmse.rank(),
                                    ab.regret.rank())[0])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        out.append(dict(mech=mech, rate=rate, tau_rmse_vs_regret=tau,
                        tau_ci_lo=lo, tau_ci_hi=hi, p_value=p,
                        rho_rmse_vs_regret=rho, tau_rmse_vs_phm08=tau_phm))
    return pd.DataFrame(out)


divergence = ranking_divergence(results, dataset=ANCHOR)
print(f"=== H1 divergence on {ANCHOR} "
      "(low tau vs regret, high tau vs PHM08 = decision-layer story) ===")
print(divergence.round(3).to_string(index=False))


def h2_wilcoxon(df, dataset=None, a="csdi", b="saits"):
    """One-sided paired Wilcoxon per (mech, rate): H2 pre-registers the
    direction (Rubin-pooled CSDI has LOWER risk-aware regret)."""
    if dataset:
        df = df[df.dataset == dataset]
    out = []
    for (mech, rate), g in df.groupby(["mech", "rate"]):
        av = g[g.imputer == a].sort_values("seed").regret_risk.values
        bv = g[g.imputer == b].sort_values("seed").regret_risk.values
        if len(av) >= 5 and (av != bv).any():
            stat, p = wilcoxon(av, bv, alternative="less")
        else:
            stat, p = np.nan, np.nan
        out.append(dict(mech=mech, rate=rate, csdi_mean=av.mean(),
                        saits_mean=bv.mean(), one_sided_p=p))
    return pd.DataFrame(out)


h2 = h2_wilcoxon(results, dataset=ANCHOR)
print(f"\n=== H2 one-sided Wilcoxon on {ANCHOR} (n={len(SEEDS)} seeds) ===")
print(h2.round(4).to_string(index=False))


def seed_ci(df, dataset=None, metric="regret_risk", z=1.96):
    if dataset:
        df = df[df.dataset == dataset]
    g = df.groupby(["imputer", "seed"])[metric].mean().reset_index()
    agg = g.groupby("imputer")[metric].agg(["mean", "std", "count"])
    agg["ci95"] = z * agg["std"] / np.sqrt(agg["count"])
    return agg.round(3)


print(f"\n=== Seed-level 95% CIs on {ANCHOR} (risk-aware regret) ===")
print(seed_ci(results, dataset=ANCHOR))


def repair_for_the_twin(df, dataset=None):
    if dataset:
        df = df[df.dataset == dataset]
    real = df[df.imputer.isin(REAL_IMPUTERS)]
    agg = real.groupby("imputer")[["imp_rmse", "regret_risk"]].mean()
    by_rmse, by_twin = agg.imp_rmse.idxmin(), agg.regret_risk.idxmin()
    return dict(dataset=dataset or "all",
                selected_by_rmse=by_rmse,
                cost_of_rmse_choice=float(agg.loc[by_rmse, "regret_risk"]),
                selected_by_twin=by_twin,
                cost_of_twin_choice=float(agg.loc[by_twin, "regret_risk"]),
                saving_pct=float(100 * (1 - agg.loc[by_twin, "regret_risk"]
                                        / agg.loc[by_rmse, "regret_risk"])))


print("\n=== H3 repair-for-the-twin criterion (per dataset) ===")
for _ds in SUBSETS_TO_RUN:
    print(repair_for_the_twin(results, dataset=_ds))


print("\n=== True B-term ablation (MI imputers; harness-computed) ===")
abl = (results[results.imputer.isin(["mice", "csdi"])]
       .groupby(["dataset", "imputer"])[
           ["coverage", "coverage_noB", "width", "width_noB",
            "regret_risk", "regret_risk_noB"]].mean().round(3))
print(abl.to_string())

---
## Section 7 — Publication Figure Suite (300 DPI, colorblind-safe, saved to Drive)

The manuscript exhibit set, in narrative order:
**F1** divergence scatter with decision-danger quadrant · **F2** τ heatmap across mechanism × rate with
bootstrap CIs · **F3** rank bump chart · **F4** coverage-vs-width with the conformal target ·
**F5** true B-term ablation dumbbells · **F6** cost-ratio sensitivity maps (winner + CSDI advantage,
computed post-hoc from archived predictions) · **F7** decision-cost decomposition (missed failures vs
false alarms) · **F8** signed-error distributions with the "late = penalised" region shaded ·
**F9** twin convergence. Every PDF is written to `figures/` on Drive.

In [ ]:
# ============================================================
# Section 7a: Style + F1-F4 core exhibits
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.family": "serif",
    "text.usetex": False, "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
CB = sns.color_palette("colorblind")
IMP_COLOR = {imp: CB[i % len(CB)] for i, imp in enumerate(IMPUTERS)}


def _save(fig, name):
    fig.savefig(FIGS_DIR / f"{name}.pdf", bbox_inches="tight")
    print(f"[fig] saved {name}.pdf -> Drive")


def f1_divergence_scatter(df, dataset=ANCHOR, mech="MNAR", rate=0.5):
    sub = (df[(df.dataset == dataset) & (df.mech == mech)
              & (df.rate == rate) & df.imputer.isin(REAL_IMPUTERS)]
           .groupby("imputer")[["imp_rmse", "regret"]].mean())
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    mx, my = sub.imp_rmse.median(), sub.regret.median()
    ax.add_patch(Rectangle((ax.get_xlim()[0], 0), 0, 0))  # init limits
    ax.axvline(mx, color="grey", lw=0.8, ls=":")
    ax.axhline(my, color="grey", lw=0.8, ls=":")
    for name, r in sub.iterrows():
        ax.scatter(r.imp_rmse, r.regret, s=110, color=IMP_COLOR[name],
                   edgecolor="black", lw=0.5, zorder=3)
        ax.annotate(name, (r.imp_rmse, r.regret), fontsize=9,
                    xytext=(6, 5), textcoords="offset points")
    # danger quadrant: good reconstruction, bad decisions
    ax.text(0.02, 0.97, "good repair,\nbad decisions",
            transform=ax.transAxes, fontsize=8, va="top", color="darkred")
    ax.set_xlabel("Imputation RMSE  (lower = better repair)")
    ax.set_ylabel("Decision regret  (lower = better twin)")
    ax.set_title(f"{dataset}: repair quality vs twin decisions "
                 f"({mech}, {int(rate*100)}%)")
    fig.tight_layout()
    _save(fig, f"F1_divergence_{dataset}_{mech}_{int(rate*100)}")
    return fig


def f2_tau_heatmap(div, dataset=ANCHOR):
    piv = div.pivot(index="mech", columns="rate",
                    values="tau_rmse_vs_regret")
    annot = div.pivot(index="mech", columns="rate", values="tau_ci_lo")
    labels = np.empty(piv.shape, dtype=object)
    for i, m in enumerate(piv.index):
        for j, r in enumerate(piv.columns):
            row = div[(div.mech == m) & (div.rate == r)].iloc[0]
            labels[i, j] = (f"{row.tau_rmse_vs_regret:.2f}\n"
                            f"[{row.tau_ci_lo:.2f}, {row.tau_ci_hi:.2f}]")
    fig, ax = plt.subplots(figsize=(5.4, 3.6))
    sns.heatmap(piv, annot=labels, fmt="", cmap="RdYlGn", center=0.5,
                vmin=-1, vmax=1, cbar_kws={"label": r"Kendall $\tau$"},
                linewidths=0.5, ax=ax, annot_kws={"fontsize": 8})
    ax.set_title(f"{dataset}: $\\tau$(imputation RMSE, decision regret)\n"
                 "with seed-bootstrap 95% CIs "
                 "(green $\\approx$ 1 = rankings agree)")
    ax.set_xlabel("Missingness rate")
    ax.set_ylabel("Mechanism")
    fig.tight_layout()
    _save(fig, f"F2_tau_heatmap_{dataset}")
    return fig


def f3_rank_bump(df, dataset=ANCHOR, mech="MNAR", rate=0.5):
    sub = (df[(df.dataset == dataset) & (df.mech == mech)
              & (df.rate == rate) & df.imputer.isin(REAL_IMPUTERS)]
           .groupby("imputer")[["imp_rmse", "regret"]].mean())
    r1, r2 = sub.imp_rmse.rank(), sub.regret.rank()
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    for name in sub.index:
        ax.plot([0, 1], [r1[name], r2[name]], "-o",
                color=IMP_COLOR[name], lw=2)
        ax.annotate(name, (1.03, r2[name]), fontsize=9, va="center")
        ax.annotate(name, (-0.03, r1[name]), fontsize=9, va="center",
                    ha="right")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["rank by\nimputation RMSE",
                        "rank by\ndecision regret"])
    ax.set_ylabel("Rank (1 = best)")
    ax.invert_yaxis()
    ax.set_xlim(-0.45, 1.45)
    ax.set_title(f"{dataset}: crossings = decision-layer divergence")
    fig.tight_layout()
    _save(fig, f"F3_bump_{dataset}_{mech}")
    return fig


def f4_coverage_width(df, dataset=ANCHOR):
    sub = (df[df.dataset == dataset]
           .groupby("imputer")[["coverage", "width"]].mean())
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    for name, r in sub.iterrows():
        ax.scatter(r.width, r.coverage, s=110, color=IMP_COLOR[name],
                   edgecolor="black", lw=0.5, zorder=3)
        ax.annotate(name, (r.width, r.coverage), fontsize=8,
                    xytext=(5, 4), textcoords="offset points")
    ax.axhline(NOMINAL, ls="--", color="grey",
               label=f"nominal {int(NOMINAL*100)}% (clean-conformal target)")
    ax.set_xlabel("Mean interval width")
    ax.set_ylabel("Empirical coverage under corruption")
    ax.set_title(f"{dataset}: calibration under corruption")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _save(fig, f"F4_coverage_width_{dataset}")
    return fig


f1_divergence_scatter(results)
f2_tau_heatmap(divergence)
f3_rank_bump(results)
f4_coverage_width(results)
plt.show()

In [ ]:
# ============================================================
# Section 7b: F5 true B-ablation + F6 cost-sensitivity (from preds)
# ============================================================
def f5_ablation_dumbbell(df, dataset=ANCHOR):
    sub = (df[(df.dataset == dataset) & df.imputer.isin(["mice", "csdi"])]
           .groupby("imputer")[["coverage", "coverage_noB",
                                "regret_risk", "regret_risk_noB"]].mean())
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.4))
    for ax, (a, b, lab) in zip(axes, [
            ("coverage_noB", "coverage", "Interval coverage"),
            ("regret_risk_noB", "regret_risk", "Risk-aware regret")]):
        for y, (name, r) in enumerate(sub.iterrows()):
            ax.plot([r[a], r[b]], [y, y], "-", color="grey", lw=2, zorder=1)
            ax.scatter(r[a], y, s=90, color=CB[7], zorder=3,
                       label="without B" if y == 0 else None)
            ax.scatter(r[b], y, s=90, color=IMP_COLOR[name], zorder=3,
                       edgecolor="black", lw=0.5,
                       label="with B (Rubin)" if y == 0 else None)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub.index)
        ax.set_xlabel(lab)
        if lab.startswith("Interval"):
            ax.axvline(NOMINAL, ls="--", color="grey", lw=0.8)
        ax.legend(fontsize=8)
    fig.suptitle(f"{dataset}: removing the between-imputation term $B$ "
                 "collapses coverage and inflates decision cost", y=1.04)
    fig.tight_layout()
    _save(fig, f"F5_ablation_{dataset}")
    return fig


def _load_anchor_preds(dataset=ANCHOR, mech="MNAR", rate=0.3):
    packs = []
    for seed in SEEDS:
        p = preds_path(dataset, mech, rate, seed)
        if p.exists():
            packs.append(np.load(p))
    return packs


def f6_cost_sensitivity(dataset=ANCHOR, mech="MNAR", rate=0.3,
                        ccm_grid=(2, 5, 10, 20, 50),
                        tau_grid=(10, 15, 20, 25, 30)):
    """Post-hoc from archived predictions: (i) which imputer the twin
    criterion selects across the cost landscape; (ii) CSDI-Rubin advantage
    over SAITS-point. No re-running required."""
    packs = _load_anchor_preds(dataset, mech, rate)
    q = CONFORMAL_Q[dataset]
    reg = {imp: np.zeros((len(ccm_grid), len(tau_grid)))
           for imp in REAL_IMPUTERS}
    for pk in packs:
        y = pk["y"]
        for imp in REAL_IMPUTERS:
            mu, var = pk[f"{imp}_mean"], pk[f"{imp}_var"]
            for i, ccm in enumerate(ccm_grid):
                cost = dict(C_pm=1.0, C_cm=float(ccm), C_fa=2.0)
                for j, tau in enumerate(tau_grid):
                    reg[imp][i, j] += risk_aware_regret(
                        mu, var, y, q, tau=tau, cost=cost)
    stack = np.stack([reg[imp] for imp in REAL_IMPUTERS])   # [I, C, T]
    winner_idx = stack.argmin(axis=0)
    adv = ((reg["saits"] - reg["csdi"])
           / np.maximum(reg["saits"], 1e-9) * 100)

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
    im = axes[0].imshow(winner_idx, cmap="tab10", vmin=0,
                        vmax=len(REAL_IMPUTERS) - 1, aspect="auto")
    for i in range(winner_idx.shape[0]):
        for j in range(winner_idx.shape[1]):
            axes[0].text(j, i, REAL_IMPUTERS[winner_idx[i, j]],
                         ha="center", va="center", fontsize=7,
                         color="white", weight="bold")
    axes[0].set_title("Twin-criterion winner across cost landscape")
    sns.heatmap(adv, annot=True, fmt=".0f", cmap="BuGn", ax=axes[1],
                cbar_kws={"label": "% cost reduction"},
                xticklabels=tau_grid, yticklabels=ccm_grid)
    axes[1].set_title("CSDI-Rubin saving vs SAITS-point (%)")
    for ax in axes:
        ax.set_xlabel(r"alarm threshold $\tau$ (cycles)")
        ax.set_ylabel(r"failure/planned cost ratio $C_{cm}/C_{pm}$")
    axes[0].set_xticks(range(len(tau_grid)))
    axes[0].set_xticklabels(tau_grid)
    axes[0].set_yticks(range(len(ccm_grid)))
    axes[0].set_yticklabels(ccm_grid)
    fig.suptitle(f"{dataset} ({mech}, {int(rate*100)}%): "
                 "conclusions across operator cost structures", y=1.03)
    fig.tight_layout()
    _save(fig, f"F6_cost_sensitivity_{dataset}")
    return fig


f5_ablation_dumbbell(results)
f6_cost_sensitivity()
plt.show()

In [ ]:
# ============================================================
# Section 7c: F7 regret decomposition, F8 error violins, F9 convergence
# ============================================================
def f7_regret_decomposition(dataset=ANCHOR, mech="MNAR", rate=0.3,
                            tau=TAU, cost=COST):
    packs = _load_anchor_preds(dataset, mech, rate)
    q = CONFORMAL_Q[dataset]
    comp = {imp: np.zeros(3) for imp in REAL_IMPUTERS}      # [FN, FP, TP]
    for pk in packs:
        y = pk["y"]
        crit = y <= tau
        for imp in REAL_IMPUTERS:
            mu, var = pk[f"{imp}_mean"], pk[f"{imp}_var"]
            alarm = (mu - q * np.sqrt(np.maximum(var, 1e-9))) <= tau
            comp[imp] += [cost["C_cm"] * ((~alarm) & crit).sum(),
                          cost["C_fa"] * (alarm & ~crit).sum(),
                          cost["C_pm"] * (alarm & crit).sum()]
    order = sorted(REAL_IMPUTERS, key=lambda i: comp[i].sum())
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    bottoms = np.zeros(len(order))
    for k, (lab, col) in enumerate(zip(
            ["missed failures ($C_{cm}$)", "false alarms ($C_{fa}$)",
             "planned maint. ($C_{pm}$)"], [CB[3], CB[1], CB[2]])):
        vals = np.array([comp[i][k] / len(packs) for i in order])
        ax.bar(order, vals, bottom=bottoms, color=col, label=lab,
               edgecolor="white")
        bottoms += vals
    ax.set_ylabel("Mean maintenance cost per test fleet")
    ax.set_title(f"{dataset}: where each pipeline's cost comes from "
                 f"({mech}, {int(rate*100)}%, risk-aware policy)")
    ax.legend(fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    _save(fig, f"F7_cost_decomposition_{dataset}")
    return fig


def f8_error_violins(dataset=ANCHOR, mech="MNAR", rate=0.3):
    packs = _load_anchor_preds(dataset, mech, rate)
    recs = []
    for pk in packs:
        y = pk["y"]
        for imp in REAL_IMPUTERS:
            for e in (pk[f"{imp}_mean"] - y):
                recs.append(dict(imputer=imp, err=float(e)))
    dfe = pd.DataFrame(recs)
    order = (dfe.groupby("imputer").err.apply(lambda s: s.abs().mean())
             .sort_values().index)
    fig, ax = plt.subplots(figsize=(6.8, 3.8))
    ax.axhspan(0, dfe.err.max() * 1.05, color=CB[3], alpha=0.08)
    sns.violinplot(data=dfe, x="imputer", y="err", order=order,
                   hue="imputer", hue_order=order, legend=False,
                   inner="quartile", cut=0, ax=ax,
                   palette=[IMP_COLOR[i] for i in order])
    ax.axhline(0, color="black", lw=0.8)
    ax.text(0.99, 0.97, "late prediction\n(PHM08-penalised, unsafe)",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=8, color="darkred")
    ax.set_ylabel("Predicted $-$ true RUL (cycles)")
    ax.set_xlabel("")
    ax.set_title(f"{dataset}: signed-error distributions "
                 f"({mech}, {int(rate*100)}%)")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    _save(fig, f"F8_error_violins_{dataset}")
    return fig


def f9_convergence(dataset=ANCHOR):
    hist = TWIN_HISTORIES[dataset]
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    for c, (i, h) in zip(CB, enumerate(hist)):
        ax.plot(h, color=c, label=f"member {i}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation RUL RMSE")
    ax.set_title(f"{dataset}: heteroscedastic twin convergence "
                 "(MSE\u2192NLL curriculum)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _save(fig, f"F9_convergence_{dataset}")
    return fig


f7_regret_decomposition()
f8_error_violins()
f9_convergence()
plt.show()

---
## Section 8 — LaTeX Tables (booktabs, IEEE/Elsevier ready)

In [ ]:
# ============================================================
# Section 8: Copy-pasteable LaTeX tables
# ============================================================
def latex_main_table(df, dataset, caption, label):
    agg = (df[df.dataset == dataset].groupby("imputer")
           .agg(imp_rmse=("imp_rmse", "mean"), rul_rmse=("rul_rmse", "mean"),
                phm08=("phm08", "mean"), coverage=("coverage", "mean"),
                regret=("regret", "mean"),
                regret_risk=("regret_risk", "mean")).round(3))
    best = agg.loc[REAL_IMPUTERS].regret_risk.idxmin()
    body = []
    for i, r in agg.iterrows():
        nm = i.replace("_", "\\_")
        rr = (f"\\textbf{{{r.regret_risk:.1f}}}" if i == best
              else f"{r.regret_risk:.1f}")
        ir = "--" if np.isnan(r.imp_rmse) else f"{r.imp_rmse:.3f}"
        body.append(f"{nm} & {ir} & {r.rul_rmse:.2f} & {r.phm08:.1f} & "
                    f"{r.coverage:.3f} & {r.regret:.1f} & {rr} \\\\")
    return ("\\begin{table}[t]\\centering\n"
            f"\\caption{{{caption}}}\\label{{{label}}}\n"
            "\\begin{tabular}{lcccccc}\\toprule\n"
            "Imputer & Imp.\\ RMSE & RUL RMSE & PHM08 & Coverage & "
            "Regret & Risk-aware regret \\\\\\midrule\n"
            + "\n".join(body)
            + "\n\\bottomrule\\end{tabular}\\end{table}")


def latex_divergence_table(div, caption, label):
    body = "\n".join(
        f"{r.mech} & {int(r.rate*100)}\\% & "
        f"{r.tau_rmse_vs_regret:.3f} "
        f"[{r.tau_ci_lo:.2f}, {r.tau_ci_hi:.2f}] & "
        f"{r.tau_rmse_vs_phm08:.3f} & {r.p_value:.3f} \\\\"
        for _, r in div.iterrows())
    return ("\\begin{table}[t]\\centering\n"
            f"\\caption{{{caption}}}\\label{{{label}}}\n"
            "\\begin{tabular}{llccc}\\toprule\n"
            "Mech. & Rate & $\\tau$(RMSE, regret) [95\\% CI] & "
            "$\\tau$(RMSE, PHM08) & $p$ \\\\\\midrule\n"
            + body + "\n\\bottomrule\\end{tabular}\\end{table}")


def latex_ablation_table(df, caption, label):
    sub = (df[df.imputer.isin(["mice", "csdi"])]
           .groupby(["dataset", "imputer"])[
               ["coverage_noB", "coverage", "regret_risk_noB",
                "regret_risk"]].mean().round(3))
    body = "\n".join(
        f"{ds} & {imp} & {r.coverage_noB:.3f} & {r.coverage:.3f} & "
        f"{r.regret_risk_noB:.1f} & {r.regret_risk:.1f} \\\\"
        for (ds, imp), r in sub.iterrows())
    return ("\\begin{table}[t]\\centering\n"
            f"\\caption{{{caption}}}\\label{{{label}}}\n"
            "\\begin{tabular}{llcccc}\\toprule\n"
            " & & \\multicolumn{2}{c}{Coverage} & "
            "\\multicolumn{2}{c}{Risk-aware regret} \\\\\n"
            "Dataset & Imputer & w/o $B$ & with $B$ & w/o $B$ & with $B$ "
            "\\\\\\midrule\n"
            + body + "\n\\bottomrule\\end{tabular}\\end{table}")


for _ds in SUBSETS_TO_RUN:
    print(latex_main_table(results, _ds,
                           f"Repair-for-the-twin evaluation on C-MAPSS "
                           f"{_ds} (means over mechanisms, rates, "
                           f"{len(SEEDS)} seeds).", f"tab:main_{_ds}"))
    print()
print(latex_divergence_table(divergence,
                             f"Decision-layer ranking divergence on {ANCHOR} "
                             "with seed-bootstrap CIs.", "tab:divergence"))
print()
print(latex_ablation_table(results,
                           "True B-term ablation: interval quality and "
                           "decision cost with the between-imputation "
                           "variance removed.", "tab:ablation"))

---
## Section 9 — Runtime Budget, Staged Plan & Manuscript Framing

### L4 wall-clock (full settings, per dataset; all cached after first completion)
| Component | First run | Resumed run |
|---|---|---|
| Twin ensemble (5 × 60 ep, heteroscedastic) | 8–12 min | seconds (Drive) |
| SAITS / US-GAN (150 ep) / CSDI | 4–8 / 12–25 / 25–45 min | seconds (Drive) |
| Harness: 3 mech × 3 rates × **10 seeds** × 9 imputers | 45–75 min | skips finished blocks |
| **Per dataset end-to-end** | **≈ 2–3 h** | — |
| **All four subsets** | **≈ 8–12 h across sessions** | resume makes this painless |

### Pre-registered analysis decisions (state in the manuscript)
- **One-sided Wilcoxon** for H2 (direction pre-specified by the variance-decomposition argument);
  with 10 seeds the minimum attainable one-sided $p$ is $\approx 0.001$, so significance is reachable.
- **Calibration claim:** "the $B$ term recovers a large fraction of coverage lost to corruption and cuts
  risk-aware decision cost several-fold" — *not* "restores nominal coverage". F4/F5 carry this.
- **Two-layer divergence framing:** τ(RMSE, PHM08) high vs τ(RMSE, regret) low ⇒ reconstruction metrics
  fail *silently* — the failure only surfaces once a cost model is attached (F2 + Table divergence).
- **US-GAN at 150 epochs:** if it remains weak, report as GAN-instability finding with its loss trace;
  note its early-skewed errors yield mid-pack regret despite poor accuracy — divergence in the
  *opposite* direction (visible in F8).
- **Cost-structure sensitivity:** all decision conclusions are conditional on $(C_{cm}/C_{pm}, \tau)$;
  F6 shows where they hold and where they flip — include it, reviewers will ask.

### Caveats
PyPOTS API is version-sensitive (CSDI output shape asserted at runtime); Rubin's rules can be
anti-conservative for non-Bayesian imputers (the conformal multiplier is the mitigation; a bootstrap
cross-check is a good appendix); injected missingness is a controlled proxy (SWaT/WADI, Building Data
Genome 2 as external-validity follow-ups); literature bands are sanity brackets — the in-experiment
`oracle_clean` / `no_repair` bounds are the reference frame.

### Novelty positioning (Related Work paragraph)
Closest prior: **RobustUQ** (Wang et al., *RESS* 262, 2025) — single diffusion imputer + conformal UQ
for online RUL; it does not rank imputers by decision quality, does not study the reconstruction-vs-
decision ranking divergence, and does not propagate imputation posterior variance via Rubin's rules.
The "imputation accuracy ≠ downstream utility" insight exists for generic prediction (ICLR 2025
"diminishing returns"; TSI-Bench, NeurIPS 2024; arXiv:2410.06652) but not for RUL prognostics with a
maintenance cost model. Contributions: (1) the decision-layer divergence result with bootstrap-CI'd τ;
(2) Rubin-pooled uncertainty-aware repair with heteroscedastic + conformal twin intervals and the true
B-ablation; (3) the deployable **repair-for-the-twin selection criterion** with its cost-landscape map —
closing the loop DQ-Twin-Bench opened.